# NB17: Off-target evaluation (SurGen-CRC MMR + TCGA-UCEC MMR)

Applies the frozen TCGA-trained scorer (NB07) to two MMR-deficient
cohorts to assess pathway specificity. Both evaluations use the
HRD-trained model without retraining.

**Manuscript reference** (Methods + Results + Supp Table S2):
- SurGen colorectal cohort (Loeffler et al. 2024): n=330, 26 MMR-deficient
  (7.9% prevalence); AUC = 0.674 (95% CI: 0.55-0.79), AP = 0.134
- TCGA-UCEC MMR off-target: n=196, 42 MMR-deficient by MLH1/MSH2/MSH6/PMS2
  non-silent somatic mutation; AUC = 0.445 (95% CI: 0.353-0.545)

**Required env**:
- `WORKSPACE`
- `SURGEN_OPENCLIP_FEATURES_PARQUET`
- `SURGEN_MMR_LABELS_CSV`
- `MC3_MAF_PATH`

**Outputs**: `results/offtarget_mmr/{surgen.json, tcga_ucec.json,
report.json}`.

In [ ]:
import os
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import roc_auc_score, average_precision_score

# env
WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
SURGEN_OPENCLIP_FEATURES_PARQUET = Path(os.environ["SURGEN_OPENCLIP_FEATURES_PARQUET"]).resolve()
SURGEN_MMR_LABELS_CSV = Path(os.environ["SURGEN_MMR_LABELS_CSV"]).resolve()
MC3_MAF_PATH = Path(os.environ["MC3_MAF_PATH"]).resolve()
EMB_DIR = WORKSPACE / "embeddings"
LABELS_DIR = WORKSPACE / "labels"
MODELS_DIR = WORKSPACE / "models"
RESULTS_DIR = WORKSPACE / "results" / "offtarget_mmr"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
BOOT_N = 2000
MMR_GENES = ["MLH1", "MSH2", "MSH6", "PMS2"]
SILENT = {"Silent", "Intron", "3'UTR", "5'UTR", "3'Flank", "5'Flank", "IGR", "RNA"}

print(f"frozen model         : {MODELS_DIR / 'frozen_model.joblib'}")
print(f"SurGen features      : {SURGEN_OPENCLIP_FEATURES_PARQUET}")
print(f"SurGen MMR labels    : {SURGEN_MMR_LABELS_CSV}")
print(f"mc3 MAF              : {MC3_MAF_PATH}")
print(f"BOOT_N               : {BOOT_N}")


In [ ]:
frozen_path = MODELS_DIR / "frozen_model.joblib"
assert frozen_path.exists(), f"missing {frozen_path}; run NB07 first"
frozen = joblib.load(frozen_path)
pipe = frozen["pipe"]
platt = frozen["platt"]
exp_dim = int(frozen["feat_dim"])

def predict_prob(X_arr):
    z = pipe.predict(X_arr).reshape(-1, 1)
    return platt.predict_proba(z)[:, 1]

def boot_ci(y, p, fn, n_boot=BOOT_N, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y); vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yb, pb = y[idx], p[idx]
        if yb.sum() == 0 or yb.sum() == n: continue
        try: vals.append(fn(yb, pb))
        except Exception: pass
    if not vals: return (float("nan"), float("nan"))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

print(f"frozen model loaded; expects {exp_dim}-d input")


In [ ]:
# SurGen-CRC: HRD-trained scorer, off-target MMR outcome
print("\n=== SurGen-CRC ===")
X = pd.read_parquet(SURGEN_OPENCLIP_FEATURES_PARQUET)
if "patient" in X.columns:
    X = X.set_index("patient")
X.index = X.index.astype(str).str.upper().str.strip()

L = pd.read_csv(SURGEN_MMR_LABELS_CSV)
L.columns = [str(c) for c in L.columns]
nm = {re.sub(r"[^a-z0-9]+", "_", c.lower()): c for c in L.columns}
pat_col = nm.get("patient") or nm.get("patient_id") or nm.get("case_id")
mmr_col = (nm.get("mmr_deficient") or nm.get("mmrd") or nm.get("mmr_status")
           or nm.get("msi_status") or nm.get("mmr") or nm.get("y") or nm.get("label"))
assert pat_col and mmr_col, f"need patient + MMR label columns; got {list(L.columns)}"

def to_mmr_deficient(v):
    s = str(v).strip().lower()
    if s in {"1", "yes", "true", "deficient", "mmr-d", "mmrd", "msi-h", "msi_high"}: return 1
    if s in {"0", "no", "false", "proficient", "mmr-p", "mmrp", "mss", "msi_stable"}: return 0
    try: return int(float(v))
    except Exception: return np.nan

L["_patient"] = L[pat_col].astype(str).str.upper().str.strip()
L["_y"] = L[mmr_col].map(to_mmr_deficient)
L = L.dropna(subset=["_patient", "_y"]).drop_duplicates("_patient").set_index("_patient")
L["_y"] = L["_y"].astype(int)

common = sorted(set(X.index) & set(L.index))
X = X.loc[common]; y = L.loc[common, "_y"].astype(int).values

if X.shape[1] != exp_dim:
    surgen = {"error": f"feature dim {X.shape[1]} != frozen model expects {exp_dim}"}
elif y.sum() == 0 or y.sum() == len(y):
    surgen = {"error": f"single-class labels (n_pos={int(y.sum())} of {len(y)})"}
else:
    p = predict_prob(X.values.astype(np.float32))
    auc = float(roc_auc_score(y, p))
    ap = float(average_precision_score(y, p))
    auc_lo, auc_hi = boot_ci(y, p, roc_auc_score)
    ap_lo, ap_hi = boot_ci(y, p, average_precision_score)
    surgen = {
        "n": int(len(y)), "n_pos": int(y.sum()), "n_neg": int(len(y) - y.sum()),
        "prevalence": float(y.mean()),
        "AUROC": auc, "AUROC_lo": auc_lo, "AUROC_hi": auc_hi,
        "AP": ap, "AP_lo": ap_lo, "AP_hi": ap_hi,
    }
    print(f"  n={surgen['n']}  MMR-d={surgen['n_pos']} ({100*surgen['prevalence']:.1f}%)")
    print(f"  AUROC = {auc:.3f} ({auc_lo:.3f}-{auc_hi:.3f})  AP = {ap:.3f}")
    print(f"  manuscript ref: AUC = 0.674 (0.55-0.79), AP = 0.134")

(RESULTS_DIR / "surgen.json").write_text(json.dumps(surgen, indent=2))


In [ ]:
# TCGA-UCEC MMR off-target
print("\n=== TCGA-UCEC MMR off-target ===")

labels = pd.read_parquet(LABELS_DIR / "labels.parquet")
labels["patient"] = labels["patient"].astype(str).str.upper().str.slice(0, 12)
labels = labels.drop_duplicates("patient").set_index("patient")
ucec_pts = labels.index[labels["cancer"] == "UCEC"]
print(f"TCGA-UCEC patients in labels: {len(ucec_pts)}")

X_tcga = pd.read_parquet(EMB_DIR / "patient_means_clean.parquet").set_index("patient")
X_tcga.index = X_tcga.index.astype(str).str.upper().str.slice(0, 12)


In [ ]:
print("reading mc3 MAF (this may take a minute) ...")
maf = pd.read_csv(
    MC3_MAF_PATH, sep="\t", dtype=str, comment="#", low_memory=False,
    usecols=["Hugo_Symbol", "Variant_Classification", "Tumor_Sample_Barcode"]
)
maf["patient"] = maf["Tumor_Sample_Barcode"].astype(str).str.upper().str.slice(0, 12)
maf_ns = maf.loc[~maf["Variant_Classification"].isin(SILENT)].copy()

mmr_pts = set()
for g in MMR_GENES:
    pts_g = set(maf_ns.loc[maf_ns["Hugo_Symbol"] == g, "patient"])
    mmr_pts |= pts_g
    print(f"  {g}: {len(pts_g)} TCGA patients with non-silent mutations")

candidate = sorted(set(ucec_pts) & set(X_tcga.index))
print(f"UCEC patients with WSI features: {len(candidate)}")
y_ucec = np.array([1 if p in mmr_pts else 0 for p in candidate], dtype=int)
print(f"MMR-deficient by mutation: {int(y_ucec.sum())}  (manuscript ref: 42)")
print(f"total UCEC evaluable     : {len(y_ucec)}      (manuscript ref: 196)")


In [ ]:
X_ucec = X_tcga.loc[candidate].values.astype(np.float32)
if X_ucec.shape[1] != exp_dim:
    ucec_res = {"error": f"feature dim {X_ucec.shape[1]} != frozen model expects {exp_dim}"}
elif y_ucec.sum() == 0 or y_ucec.sum() == len(y_ucec):
    ucec_res = {"error": f"single-class labels (n_pos={int(y_ucec.sum())} of {len(y_ucec)})"}
else:
    p_ucec = predict_prob(X_ucec)
    auc = float(roc_auc_score(y_ucec, p_ucec))
    ap = float(average_precision_score(y_ucec, p_ucec))
    auc_lo, auc_hi = boot_ci(y_ucec, p_ucec, roc_auc_score)
    ap_lo, ap_hi = boot_ci(y_ucec, p_ucec, average_precision_score)
    ucec_res = {
        "n": int(len(y_ucec)),
        "n_pos": int(y_ucec.sum()), "n_neg": int(len(y_ucec) - y_ucec.sum()),
        "AUROC": auc, "AUROC_lo": auc_lo, "AUROC_hi": auc_hi,
        "AP": ap, "AP_lo": ap_lo, "AP_hi": ap_hi,
    }
    print(f"  AUROC = {auc:.3f} ({auc_lo:.3f}-{auc_hi:.3f})  AP = {ap:.3f}")
    print(f"  manuscript ref: AUC = 0.445 (0.353-0.545)")

(RESULTS_DIR / "tcga_ucec.json").write_text(json.dumps(ucec_res, indent=2))


In [ ]:
report = {
    "seed": SEED,
    "bootstrap_n": BOOT_N,
    "frozen_model_path": str(frozen_path),
    "surgen": surgen,
    "tcga_ucec": ucec_res,
    "manuscript_refs": {
        "SurGen": {"n": 330, "n_pos": 26, "prevalence_pct": 7.9,
                   "AUROC": 0.674, "AUROC_CI": [0.55, 0.79], "AP": 0.134},
        "TCGA_UCEC_MMR": {"n": 196, "n_pos": 42,
                          "AUROC": 0.445, "AUROC_CI": [0.353, 0.545]},
    },
}
(RESULTS_DIR / "report.json").write_text(json.dumps(report, indent=2, default=str))
print(json.dumps(report, indent=2, default=str))
